# Unit 6, Lecture 1: Responsible AI

Units 1 to 5 asked **can it work?** This unit asks **should it act?** A capable
agent can still be wrong to deploy: it can decide unfairly, act without
explanation, and leave no trace.

Responsible AI rests on three principles, and the danger is treating them as a
poster on the wall. This notebook does the opposite: it turns each into a **check
your agent can run on itself**, because a principle you cannot check is one you
cannot keep.

Think of a good loan officer: (1) treats similar people the same, (2) can explain
every decision, (3) leaves a record. That is fairness, transparency,
accountability. Everything here runs offline.

## 1. Fairness: flip who they are, the answer must not move

Fairness has a precise, testable meaning: hold everything relevant fixed, change
only the protected attribute, and the decision must not change.

In [ ]:
from cse476.responsible import decide, is_fair

# the group argument exists so we can flip it; the decision must ignore it
print("same person, group A:", decide(40000, 700, applicant_group="A"))
print("same person, group B:", decide(40000, 700, applicant_group="B"))
print()
print("fair across groups?", is_fair(decide, 40000, 700, ["A", "B", "C"]))

### The test earns its keep: it catches a real bias

A subtle bias, holding one group to a higher bar, is invisible in a single
decision but caught instantly by the flip test, which also names who differed.

In [ ]:
from cse476.responsible import is_fair, find_unfairness

def biased(income, credit_score, applicant_group=None):
    bar = 750 if applicant_group == "B" else 650   # group B held to a higher bar
    return "approve" if (income >= 30000 and credit_score >= bar) else "decline"

print("is the biased rule fair?", is_fair(biased, 40000, 700, ["A", "B"]))
print("who was treated differently?", find_unfairness(biased, 40000, 700, ["A", "B"]))

## 2. Transparency: every decision carries its reason

A decision with no reason is a black box. The reason is built from the **same
thresholds** the decision uses, so it can never drift from the real logic.

In [ ]:
from cse476.responsible import decide_with_reason

v = decide_with_reason(25000, 700)
print("decision:", v["decision"])
print("because:")
for r in v["because"]:
    print("  -", r)

## 3. Accountability: leave a record that can be reviewed

Fairness and transparency are about the decision **now**. Accountability is about
**later**, when someone asks why. The record needs who, what, and why.

In [ ]:
from cse476.responsible import decision_record

rec = decision_record("loan-agent-v1", 25000, 700)
print("agent:  ", rec.agent)      # WHO decided
print("inputs: ", rec.inputs)     # on WHAT
print("outcome:", rec.outcome)    # WHAT it decided
print("reason: ", rec.reason)     # and WHY
print("reviewable?", rec.is_reviewable())
print()

# a record missing who/what/why is not reviewable:
from cse476.responsible import DecisionRecord
empty = DecisionRecord(agent="", inputs={}, outcome="decline", reason=[])
print("empty record reviewable?", empty.is_reviewable())

## Why this matters more for an agent

In [ ]:
from cse476.responsible import RESPONSIBLE_MAP, why_responsible

for concept, meaning in RESPONSIBLE_MAP.items():
    print(f"{concept:28} ->  {meaning}")
print()
for k, v in why_responsible().items():
    print(f"{k:22}: {v}")

A model gives one answer; an agent takes ten thousand actions a day,
automatically. A biased rule in a chatbot annoys one person; the same rule in an
agent harms ten thousand before anyone notices. That scale is why the checks must
live in the code, not in a human glancing at the output.

## Your turn

**1. Fairness-test a decision.** Add a protected attribute your capstone agent
should ignore, then write the flip test. If the decision moves, you found bias.

**2. Attach a reason.** Make one decision return its reason alongside the outcome,
built from the same rule.

**3. Find the unaccountable action.** Name one action that leaves no record. If
someone asked why it did that last Tuesday, could you answer?

In [ ]:
# your work here
